<a href="https://colab.research.google.com/github/kynodontas-flac/MetodosDigitales_2026_2/blob/main/9_Web%20Scraping/9_Minado_YT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minería de datos de redes sociales
#
#
#
#### YouTube es otra gran opción para recuperar grandes cantidades de datos de redes sociales
#
#
#
#### A continuación tenemos 2 usos muy interesantes: 1) recuperar los videos en una búsqueda de palabras claves con datos como número de comentarios, me gusta, suscriptores del canal, nombre del canal, y 2) los comentarios de un video en concreto con el usuario que los hizo y el conteo de me gusta.

In [15]:
!pip install google-api-python-client

import numpy as np
import pandas as pd
from google.colab import userdata
import googleapiclient.discovery

Para usar la API de datos de YouTube, necesitarás una clave de API. Si aún no tienes una, créala en Google Cloud Console y habilita la API de datos de YouTube v3.

En Colab, agrega tu clave de API al administrador de secretos debajo de la "🔑" en el panel izquierdo. Asígnale el nombre `YOUTUBE_API_KEY`.

In [16]:
# Cargar la clave API desde los secretos de Colab
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY')

Ahora, aquí tienes un script simple para buscar videos en YouTube usando una consulta con palabras clave. La función `youtube_search` devolverá una lista de detalles de videos, incluyendo sus IDs, títulos y descripciones.

In [27]:
def youtube_search(query, api_key, total_max_results=500):
    youtube = googleapiclient.discovery.build(
        "youtube", "v3", developerKey=api_key)

    video_ids = []
    channel_ids = set()
    raw_videos_data = {}
    next_page_token = None
    retrieved_count = 0

    while retrieved_count < total_max_results:
        # 1. Búsqueda inicial de videos con paginación
        search_request = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=min(500, total_max_results - retrieved_count), # Máximo 50 resultados por página
            pageToken=next_page_token
        )
        search_response = search_request.execute()

        for item in search_response.get("items", []):
            if retrieved_count >= total_max_results:
                break
            video_id = item['id']['videoId']
            channel_id = item['snippet']['channelId']
            video_ids.append(video_id)
            channel_ids.add(channel_id)
            raw_videos_data[video_id] = {
                'title': item['snippet']['title'],
                'description': item['snippet']['description'],
                'channel_title': item['snippet']['channelTitle'],
                'channel_id': channel_id
            }
            retrieved_count += 1

        next_page_token = search_response.get('nextPageToken')
        if not next_page_token or retrieved_count >= total_max_results:
            break

    # Si no se encuentran videos, devolver una lista vacía
    if not video_ids:
        return []

    # 2. Obtener estadísticas de video (me gusta y número de comentarios)|
    video_statistics = {}
    # La API de videos 'list' acepta hasta 50 IDs de video a la vez.
    # Si tenemos más de 50 IDs, necesitamos procesarlos en lotes.
    batch_size = 50
    for i in range(0, len(video_ids), batch_size):
        batch_video_ids = video_ids[i:i + batch_size]
        video_stats_request = youtube.videos().list(
            part="statistics",
            id=",".join(batch_video_ids)
        )
        video_stats_response = video_stats_request.execute()
        video_statistics.update({item['id']: item['statistics'] for item in video_stats_response.get("items", [])})

    # 3. Obtener estadísticas de canal (suscriptores)
    channel_statistics = {}
    # De manera similar, procesar las solicitudes de estadísticas de canal en lotes
    channel_ids_list = list(channel_ids)
    for i in range(0, len(channel_ids_list), batch_size):
        batch_channel_ids = channel_ids_list[i:i + batch_size]
        channel_stats_request = youtube.channels().list(
            part="statistics",
            id=",".join(batch_channel_ids) # Convertir conjunto a lista para join
        )
        channel_stats_response = channel_stats_request.execute()
        channel_statistics.update({item['id']: item['statistics'] for item in channel_stats_response.get("items", [])})

    # 4. Combinar datos
    final_videos = []
    for video_id in video_ids:
        video_data = raw_videos_data.get(video_id, {})
        stats = video_statistics.get(video_id, {})
        channel_stats = channel_statistics.get(video_data.get('channel_id'), {})

        final_videos.append({
            'video_id': video_id,
            'title': video_data.get('title', 'N/A'),
            'description': video_data.get('description', 'N/A'),
            'channel_title': video_data.get('channel_title', 'N/A'), # Nombre del canal (handle de usuario)
            'likes': stats.get('likeCount', '0'), # Por defecto '0' si no se encuentra
            'subscribers': channel_stats.get('subscriberCount', '0'), # Por defecto '0' si no se encuentra
            'comment_count': stats.get('commentCount', '0') # Número de comentarios
        })
    return final_videos

# Ejemplo de uso:
search_query = "Minería de datos en redes sociales"
results = youtube_search(search_query, YOUTUBE_API_KEY, total_max_results=500)

print(f"Se encontraron {len(results)} videos para '{search_query}':")
for i, video in enumerate(results):
    if i >= 4: # Imprimir solo los primeros 4 videos
        break
    print(f"\n--- Video {i+1} ---")
    print(f"Título: {video['title']}")
    print(f"ID de Video: {video['video_id']}")
    print(f"Nombre del Canal (Handle de Usuario): {video['channel_title']}")
    print(f"Me Gusta: {video['likes']}")
    print(f"Suscriptores: {video['subscribers']}")
    print(f"Número de Comentarios: {video['comment_count']}") # Añadido el número de comentarios
    print(f"Descripción: {video['description']}")
    print(f"Ver en YouTube: https://www.youtube.com/watch?v={video['video_id']}")

Se encontraron 500 videos para 'Minería de datos en redes sociales':

--- Video 1 ---
Título: Qué es minería de datos? | Qué es?
ID de Video: SrhQ0wWFg4Q
Nombre del Canal (Handle de Usuario): Qué es ?
Me Gusta: 149
Suscriptores: 2260
Número de Comentarios: 6
Descripción: data #datamining #educacion Imagina un mundo en el que cada clic, cada "me gusta", cada comentario e incluso cada compra ...
Ver en YouTube: https://www.youtube.com/watch?v=SrhQ0wWFg4Q

--- Video 2 ---
Título: Minería de Datos
ID de Video: Kzs9bXRb_6A
Nombre del Canal (Handle de Usuario): Colegio de Contadores Públicos de México
Me Gusta: 5
Suscriptores: 38600
Número de Comentarios: 2
Descripción: Conoce la importancia de la minería de datos en la detección de fraudes en las empresas y las recomendaciones que los ...
Ver en YouTube: https://www.youtube.com/watch?v=Kzs9bXRb_6A

--- Video 3 ---
Título: Minería de Datos en Redes Sociales
ID de Video: ohwfdAxWtIw
Nombre del Canal (Handle de Usuario): Jaime Huarsaya Rivera


In [28]:
# Convertir la lista de diccionarios (results) en un DataFrame de pandas
df_youtube_results = pd.DataFrame(results)

# Añadir la columna con el enlace de YouTube
df_youtube_results['youtube_link'] = df_youtube_results['video_id'].apply(lambda x: f"https://www.youtube.com/watch?v={x}")

# Mostrar el DataFrame
print("DataFrame de Resultados de Búsqueda de YouTube:")
display(df_youtube_results)

DataFrame de Resultados de Búsqueda de YouTube:


,video_id,title,description,channel_title,likes,subscribers,comment_count,youtube_link
0,SrhQ0wWFg4Q,Qué es minería de datos? | Qué es?,data #datamining #educacion Imagina un mundo e...,Qué es ?,149,2260,6,https://www.youtube.com/watch?v=SrhQ0wWFg4Q
1,Kzs9bXRb_6A,Minería de Datos,Conoce la importancia de la minería de datos e...,Colegio de Contadores Públicos de México,5,38600,2,https://www.youtube.com/watch?v=Kzs9bXRb_6A
2,ohwfdAxWtIw,Minería de Datos en Redes Sociales,Las técnicas de Minería de Datos que debes usa...,Jaime Huarsaya Rivera,6,10,2,https://www.youtube.com/watch?v=ohwfdAxWtIw
3,hyyWfuTG1TA,Minería de datos sobre streams de redes sociales,,Maria Romero,0,0,0,https://www.youtube.com/watch?v=hyyWfuTG1TA
4,qat9EBUzu74,Minería de datos en las redes sociales. Danna ...,,Danna Valentina Jaramillo Zapata,2,0,0,https://www.youtube.com/watch?v=qat9EBUzu74
...,...,...,...,...,...,...,...,...
495,e1zFu-oKh0k,Extracción de Datos de Producción Agrícola 202...,Quieres aprender a extraer datos de producción...,Nelson Arauco,86,1030,6,https://www.youtube.com/watch?v=e1zFu-oKh0k
496,XXHpV3-2-TU,EL COLAPSO DE LA MINERÍA DE TIERRAS RARAS EN E...,Únete conviértete en miembro del canal para ac...,IngMorrison,234,293000,30,https://www.youtube.com/watch?v=XXHpV3-2-TU
497,3BFJ0Av1vvs,Una Máquina Para Minar Criptomonedas (Helium $...,En este video te cuento mi experiencia en la m...,HackWise,243,268000,26,https://www.youtube.com/watch?v=3BFJ0Av1vvs
498,jfnWRWV0pKQ,Experto en minería: &quot;Muchas veces lo que ...,"Carlos Salazar, experto en minería, recalca qu...",Eco Tv Panama,15,54700,9,https://www.youtube.com/watch?v=jfnWRWV0pKQ


In [44]:
df_youtube_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       500 non-null    object
 1   title          500 non-null    object
 2   description    500 non-null    object
 3   channel_title  500 non-null    object
 4   likes          500 non-null    object
 5   subscribers    500 non-null    object
 6   comment_count  500 non-null    object
 7   youtube_link   500 non-null    object
dtypes: object(8)
memory usage: 31.4+ KB


In [45]:
# Convertir las columnas 'likes', 'subscribers' y 'comment_count' a tipo entero
# Primero, reemplazar cualquier valor no numérico con 0 antes de la conversión
# La función `pd.to_numeric` se usa con `errors='coerce'` para convertir valores no válidos a NaN,
# y luego `fillna(0)` para reemplazar los NaN con 0.

df_youtube_results['likes'] = pd.to_numeric(df_youtube_results['likes'], errors='coerce').fillna(0).astype(int)
df_youtube_results['subscribers'] = pd.to_numeric(df_youtube_results['subscribers'], errors='coerce').fillna(0).astype(int)
df_youtube_results['comment_count'] = pd.to_numeric(df_youtube_results['comment_count'], errors='coerce').fillna(0).astype(int)

print("Columnas 'likes', 'subscribers' y 'comment_count' convertidas a tipo entero.")
# Mostrar la información del DataFrame para verificar los tipos de datos
df_youtube_results.info()

Columnas 'likes', 'subscribers' y 'comment_count' convertidas a tipo entero.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       500 non-null    object
 1   title          500 non-null    object
 2   description    500 non-null    object
 3   channel_title  500 non-null    object
 4   likes          500 non-null    int64 
 5   subscribers    500 non-null    int64 
 6   comment_count  500 non-null    int64 
 7   youtube_link   500 non-null    object
dtypes: int64(3), object(5)
memory usage: 31.4+ KB


In [46]:
print("Estadísticas descriptivas para df_youtube_results:")
display(df_youtube_results.describe())

Estadísticas descriptivas para df_youtube_results:


,likes,subscribers,comment_count
count,5.000000e+02,5.000000e+02,500.000000
mean,1.018822e+04,2.804142e+05,174.488000
std,8.608737e+04,1.495730e+06,1592.051125
min,0.000000e+00,0.000000e+00,0.000000
25%,2.000000e+00,5.107500e+02,0.000000
50%,8.000000e+00,4.845000e+03,0.000000
75%,3.625000e+01,5.612500e+04,2.250000
max,1.580572e+06,2.510000e+07,27427.000000


In [51]:
df_youtube_results.to_csv('df_youtube_results.csv', index=False)
print("DataFrame 'df_youtube_results' guardado como 'df_youtube_results.csv'")

DataFrame 'df_youtube_results' guardado como 'df_youtube_results.csv'


In [43]:
def get_video_comments(video_id, api_key, max_results_per_page=500, total_max_comments=500):
    # Construir el servicio de la API de YouTube
    youtube = googleapiclient.discovery.build(
        "youtube", "v3", developerKey=api_key)

    comments = []
    next_page_token = None
    retrieved_count = 0

    while True and retrieved_count < total_max_comments:
        # Realizar la solicitud para obtener hilos de comentarios
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            textFormat="plainText", # Formato del texto del comentario (plainText o html)
            maxResults=min(max_results_per_page, total_max_comments - retrieved_count), # Limitar resultados por página
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response.get("items", []):
            # Extraer los detalles del comentario
            comment_snippet = item['snippet']['topLevelComment']['snippet']
            author = comment_snippet['authorDisplayName']
            text = comment_snippet['textDisplay']
            likes = comment_snippet['likeCount']

            comments.append({
                'author': author,
                'text': text,
                'likes': likes,
                'video_id': video_id # Añadir el ID del video para referencia
            })
            retrieved_count += 1
            if retrieved_count >= total_max_comments:
                break

        # Obtener el token de la siguiente página para la paginación
        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break # No hay más páginas, salir del bucle

    return comments

# También puedes probar con un ID de video específico manualmente:
manual_video_id = "dQw4w9WgXcQ" # Ejemplo de ID de video (Rick Astley - Never Gonna Give You Up)
print(f"\nRecuperando comentarios para el video ID manual: {manual_video_id}")
video_comments = get_video_comments(manual_video_id, YOUTUBE_API_KEY, total_max_comments=500)
print(f"Se encontraron {len(video_comments)} comentarios para el video {manual_video_id}:")
for i, comment in enumerate(video_comments):
    if i >= 4: # Mostrar solo los primeros 4 comentarios
        break
    print(f"\n--- Comentario {i+1} ---")
    print(f"Autor: {comment['author']}")
    print(f"Me Gusta: {comment['likes']}")
    print(f"Texto: {comment['text']}")



Recuperando comentarios para el video ID manual: dQw4w9WgXcQ
Se encontraron 500 comentarios para el video dQw4w9WgXcQ:

--- Comentario 1 ---
Autor: @YouTube
Me Gusta: 268428
Texto: can confirm: he never gave us up

--- Comentario 2 ---
Autor: @SidCostello-m4s
Me Gusta: 0
Texto: When he said “desert you” I thought he meant “dessert you” and it made me hungry😭

--- Comentario 3 ---
Autor: @tinacurlyq
Me Gusta: 1
Texto: Can someone explain what rickroll means? I’ve never heard that term and I loved this song back in the day 😊

--- Comentario 4 ---
Autor: @jazzjosh4813
Me Gusta: 0
Texto: https://www.youtube.com/watch?v=dQw4w9WgXcQ


In [42]:
# Convertir la lista de diccionarios (video_comments) en un DataFrame de pandas
df_video_comments = pd.DataFrame(video_comments)

# Mostrar el DataFrame
print("DataFrame de Comentarios de Video de YouTube:")
display(df_video_comments)

DataFrame de Comentarios de Video de YouTube:


,author,text,likes,video_id
0,@YouTube,can confirm: he never gave us up,268428,dQw4w9WgXcQ
1,@SidCostello-m4s,When he said “desert you” I thought he meant “...,0,dQw4w9WgXcQ
2,@tinacurlyq,Can someone explain what rickroll means? I’ve ...,1,dQw4w9WgXcQ
3,@jazzjosh4813,https://www.youtube.com/watch?v=dQw4w9WgXcQ,0,dQw4w9WgXcQ
4,@NoobMoons,A short lend me to this now I got ricked rolled,0,dQw4w9WgXcQ
...,...,...,...,...
495,@FanOfPBSConsolesAndStuff,"Did you know that he's originally British, but...",0,dQw4w9WgXcQ
496,@MakiMister,Just bought a hoodie With a QR code Of this. I...,0,dQw4w9WgXcQ
497,@Schlauontwitch,I got rickrolled by a QR code on Mecca chameleon,0,dQw4w9WgXcQ
498,@batuhanipekipek7345,Never Gonna Give You Up :),0,dQw4w9WgXcQ


In [47]:
df_video_comments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   author    500 non-null    object
 1   text      500 non-null    object
 2   likes     500 non-null    int64 
 3   video_id  500 non-null    object
dtypes: int64(1), object(3)
memory usage: 15.8+ KB


In [48]:
print("Estadísticas descriptivas para df_video_comments:")
display(df_video_comments.describe())

Estadísticas descriptivas para df_video_comments:


,likes
count,500.000000
mean,537.742000
std,12004.426241
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,268428.000000


In [52]:
df_video_comments.to_csv('df_video_comments.csv', index=False)
print("DataFrame 'df_video_comments' guardado como 'df_video_comments.csv'")

DataFrame 'df_video_comments' guardado como 'df_video_comments.csv'
